<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-10/NB10_interpretability.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 10: Interpretability and Transparency for Safety

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-10/lab.html>.

## Overview

Explanations can reveal spurious features, support audits and help people decide when to trust a model, but they can also mislead. This week covers the goals and evaluation of interpretability, post-hoc attribution including the author's information-geometric GEMEX package, sanity checks and mechanistic interpretability [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to state what an explanation is for and how it can be evaluated, to implement a local surrogate explanation and measure its fidelity, to run permutation importance and GEMEX on a tabular model, to perform a model randomisation sanity check, and to describe circuits, superposition, dictionary learning and probing.

## What explanations are for

Doshi-Velez and Kim defined interpretability as the ability to explain or present in understandable terms to a human, and proposed three levels of evaluation [1]. Application-grounded evaluation tests explanations with experts on the real task, human-grounded evaluation uses simplified tasks with lay people and functionally-grounded evaluation uses formal proxies without humans. For safety, explanations serve several purposes: debugging, detecting reliance on spurious features, supporting audits and appeals, and calibrating the trust that users place in a system.

Rudin argued that for high-stakes decisions, inherently interpretable models should be preferred over black boxes with post-hoc explanations, because such explanations can be unfaithful to the model they describe and because interpretable models are often nearly as accurate [2]. The volume edited by Kose, Sengoz, Chen and Marmolejo Saucedo collects methods and applications of explainable AI in healthcare, where both positions are argued in practice [4].

## Post-hoc attribution

Ribeiro, Singh and Guestrin introduced LIME, which explains one prediction by sampling perturbations around the instance, weighting them by proximity and fitting a simple interpretable model, usually a sparse linear model, to the black box's outputs [5]. The surrogate's weights form the explanation, and its fit in the neighbourhood, the fidelity, indicates how far the explanation can be trusted. Lundberg and Lee showed that several attribution methods belong to one class of additive feature attributions and that Shapley values from cooperative game theory are the unique member satisfying local accuracy, missingness and consistency; their SHAP framework estimates these values efficiently [6]. Permutation importance is a simpler global method: It measures how much performance drops when one feature's values are shuffled.

GEMEX, a model-agnostic Python package developed by Kose, takes a geometric route [3]. It treats the prediction surface of a model as a curved manifold described with Riemannian information geometry, computes feature attributions along geodesic paths instead of straight lines and reports additional quantities such as curvature, which it links to the uncertainty of the explanation. The notebook applies GEMEX in its simplest configuration and compares its ranking with permutation importance and a local surrogate. Figure 10.1 illustrates the local surrogate idea.

![Figure 10.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-10/figures/w10_fig1.png)

*Figure 10.1. A local surrogate explanation. Perturbed samples near the instance are weighted by proximity, and a weighted linear model fitted to the model's outputs gives the local feature weights [5].*

## Do explanations reflect the model?

Adebayo and colleagues proposed sanity checks for saliency maps [7]. In the model parameter randomisation test, the weights of a trained network are progressively randomised; in the data randomisation test, a model is trained on permuted labels. An explanation method that produces nearly the same output for the trained and the randomised model cannot be explaining what the model learned. Several popular methods failed these tests and behaved more like edge detectors applied to the input.

Local surrogates face a related problem of stability: Two runs with different random perturbations can yield different explanations for the same instance, as the lab shows. A careful practitioner therefore reports fidelity and stability along with the explanation and repeats sanity checks whenever a method is used in an audit.

## Mechanistic interpretability

Mechanistic interpretability tries to reverse-engineer the computations inside a network. Olah and colleagues proposed that features and the circuits connecting them are meaningful units, and they traced circuits such as curve detectors in vision models [8]. Elhage and colleagues showed with toy models that networks can represent more features than they have dimensions by storing them in superposition, which explains why many neurons respond to unrelated concepts [9]. Bricken and colleagues used dictionary learning with sparse autoencoders to decompose the activations of a small transformer into many features that were more interpretable than individual neurons [10]. Alain and Bengio introduced linear probes, classifiers trained on intermediate activations to test which information is linearly decodable [11]. A probe shows that information is present, not that the model uses it.

For AI safety, these tools promise ways to inspect internal states, for example to detect deceptive behaviour that output-based tests miss, which Week 13 discusses. Their main limitation is scale: Methods that work on small models are hard to apply comprehensively to frontier systems.

> **Pause and reflect.** A hospital asks for an explanation of every automated risk score. Which method from this week would you provide, and how would you show that the explanation is faithful?

# Hands-on lab

The notebook explains a gradient boosting model on the breast cancer data set with permutation importance, a hand-written local surrogate and GEMEX, then runs a data randomisation sanity check and trains a linear probe on the hidden layer of a small network [3, 5, 7, 11].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Model and global importance

In [ ]:
import warnings
from scipy.stats import spearmanr
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

data = load_breast_cancer()
names = np.array(data.feature_names)
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, random_state=0, stratify=data.target)
model = GradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)
print("test accuracy:", round(model.score(X_te, y_te), 3))
pi = permutation_importance(model, X_te, y_te, n_repeats=10, random_state=0)
top = np.argsort(-pi.importances_mean)[:8]
plt.barh(names[top][::-1], pi.importances_mean[top][::-1], color="#1D5B78")
plt.xlabel("drop in accuracy when shuffled"); plt.title("Permutation importance"); plt.tight_layout(); plt.show()

## 2. A local surrogate written by hand

The instance explained is the test case whose predicted probability is closest to 0.5, where the local behaviour of the model is most informative. Perturbations are sampled in standardised units around it, weighted with a Gaussian kernel and explained with a weighted ridge regression, following the idea of LIME.

In [ ]:
mu, sd = X_tr.mean(0), X_tr.std(0)
x0 = X_te[np.argmin(np.abs(model.predict_proba(X_te)[:, 1] - 0.5))]   # the test case closest to the decision boundary
rng = np.random.default_rng(SEED)
Zs = rng.normal(0, 1, size=(2000, X_tr.shape[1]))            # standardised perturbations
samples = x0 + Zs * sd * 0.5
fx = model.predict_proba(samples)[:, 1]
dist2 = np.sum((Zs * 0.5) ** 2, axis=1)
weights = np.exp(-dist2 / (2 * 0.75 ** 2 * X_tr.shape[1]))
surrogate = Ridge(alpha=1.0).fit(Zs, fx, sample_weight=weights)
local_top = np.argsort(-np.abs(surrogate.coef_))[:5]
print("prediction for the instance:", model.predict_proba(x0[None])[0, 1].round(3))
for j in local_top:
    print(f"{names[j]:25s} {surrogate.coef_[j]:+.4f}")

### Your turn, exercise 1

Compute the fidelity of the surrogate: the weighted R squared of `surrogate` on `(Zs, fx)` with `weights`. Ridge provides `score(X, y, sample_weight=...)`, but try writing it from the definition first.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _fidelity_reference():
    pred = surrogate.predict(Zs)
    ybar = np.average(fx, weights=weights)
    return 1 - np.sum(weights * (fx - pred) ** 2) / np.sum(weights * (fx - ybar) ** 2)

In [ ]:
fidelity = None  # your computation
check("Exercise 1", fidelity, _fidelity_reference())

## 3. GEMEX in its simplest form

GEMEX is installed from PyPI. The configuration below uses few geodesic steps and reference samples so that it runs in seconds; the package documentation describes richer settings and 13 plot types.

In [ ]:
!pip install -q gemex

In [ ]:
try:
    from gemex import Explainer, GemexConfig
    cfg = GemexConfig(n_geodesic_steps=8, n_reference_samples=20, interaction_order=1, verbose=False)
    exp = Explainer(model, data_type="tabular", feature_names=list(names), class_names=list(data.target_names), config=cfg)
    result = exp.explain(x0, X_reference=X_tr)
    print(result.summary())
    gemex_top = [str(f) for f, _ in result.top_features(5)]
    print("GEMEX top 5:      ", gemex_top)
    print("Local surrogate:  ", list(names[local_top]))
    print("Permutation (global):", list(names[top[:5]]))
    result.plot("gsf_bar")
except ImportError:
    print("GEMEX is not installed. Run: pip install gemex")

## 4. A sanity check by data randomisation

A model trained on shuffled labels has learned nothing about the task. If an importance ranking barely changes, the method is not explaining what the model learned.

In [ ]:
y_shuffled = np.random.default_rng(1).permutation(y_tr)
random_model = GradientBoostingClassifier(random_state=0).fit(X_tr, y_shuffled)
pi_rand = permutation_importance(random_model, X_te, y_te, n_repeats=10, random_state=0)
print("accuracy of the randomised model:", round(random_model.score(X_te, y_te), 3))

### Your turn, exercise 2

Compute the Spearman rank correlation between `pi.importances_mean` and `pi_rand.importances_mean`. A value near zero means that the explanation method passes this sanity check.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _spearman_reference():
    return spearmanr(pi.importances_mean, pi_rand.importances_mean).correlation

In [ ]:
rank_corr = None  # your computation
check("Exercise 2", rank_corr, _spearman_reference())

## 5. A linear probe

A small network is trained on the task. A probe then tests whether a concept that the network was never asked to predict, a large mean radius, is linearly decodable from its hidden layer.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(X_tr)
net = MLPClassifier(hidden_layer_sizes=(32,), max_iter=2000, random_state=0).fit(sc.transform(X_tr), y_tr)
hidden = lambda X: np.maximum(0, sc.transform(X) @ net.coefs_[0] + net.intercepts_[0])
concept_tr = X_tr[:, 0] > np.median(X_tr[:, 0]); concept_te = X_te[:, 0] > np.median(X_tr[:, 0])
probe = LogisticRegression(max_iter=2000).fit(hidden(X_tr), concept_tr)
baseline = LogisticRegression(max_iter=2000).fit(np.random.default_rng(0).normal(size=hidden(X_tr).shape), concept_tr)
print("probe accuracy on hidden activations:", round(probe.score(hidden(X_te), concept_te), 3))
print("probe accuracy on random features:  ", round(baseline.score(np.random.default_rng(1).normal(size=hidden(X_te).shape), concept_te), 3))

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-10/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. In the lab, how much did the explanation change when you resampled? What would that mean for an explanation shown to a loan applicant?
2. Compare the GEMEX, surrogate and permutation rankings from the notebook. Where do they agree, and how would you explain the differences?
3. Which safety question in your field could mechanistic interpretability help to answer, if it worked at scale?

## Weekly task and submission

Explain the predictions of a model from your field or the notebook model with at least two methods from this week. Report fidelity or stability for each, run one sanity check and write about 400 words on which explanation you would show to a domain expert and why.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**How should explanations be evaluated?.** Review evaluation approaches for explanation methods, from functionally grounded metrics to user studies and sanity checks [1, 7]. Discuss which evaluations matter for safety and include recent geometric approaches to attribution [3].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Doshi-Velez, F., & Kim, B. (2017). Towards a rigorous science of interpretable machine learning. arXiv preprint arXiv:1702.08608. <https://arxiv.org/abs/1702.08608>

[2] Rudin, C. (2019). Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead. *Nature Machine Intelligence*, 1(5), 206-215. <https://doi.org/10.1038/s42256-019-0048-x>

[3] Kose, U. (2026). *GEMEX: Geodesic Entropic Manifold Explainability (Version 1.2.2) [Python package]*. Python Package Index. <https://pypi.org/project/gemex/>

[4] Kose, U., Sengoz, N., Chen, X., & Marmolejo Saucedo, J. A. (Eds.) (2024). *Explainable Artificial Intelligence (XAI) in Healthcare*. CRC Press.

[5] Ribeiro, M. T., Singh, S., & Guestrin, C. (2016). "Why should I trust you?": Explaining the predictions of any classifier. In *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining* (pp. 1135-1144). <https://doi.org/10.1145/2939672.2939778>

[6] Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. In *Advances in Neural Information Processing Systems 30*.

[7] Adebayo, J., Gilmer, J., Muelly, M., Goodfellow, I., Hardt, M., & Kim, B. (2018). Sanity checks for saliency maps. In *Advances in Neural Information Processing Systems 31*.

[8] Olah, C., Cammarata, N., Schubert, L., Goh, G., Petrov, M., & Carter, S. (2020). Zoom in: An introduction to circuits. *Distill*, 5(3), e00024.001. <https://doi.org/10.23915/distill.00024.001>

[9] Elhage, N., Hume, T., Olsson, C., et al. (2022). Toy models of superposition. arXiv preprint arXiv:2209.10652. <https://arxiv.org/abs/2209.10652>

[10] Bricken, T., Templeton, A., Batson, J., et al. (2023). *Towards monosemanticity: Decomposing language models with dictionary learning*. Transformer Circuits Thread. <https://transformer-circuits.pub/2023/monosemantic-features>

[11] Alain, G., & Bengio, Y. (2016). Understanding intermediate layers using linear classifier probes. arXiv preprint arXiv:1610.01644. <https://arxiv.org/abs/1610.01644>